# HealthConnect ML Pipeline — Demo Notebook
## AnalystLab Africa Experience Lab — Week 5 (Machine Learning Engineering Track)

This notebook demonstrates the pipeline built in `src/` end-to-end: ingestion,
automated data quality checks, cleaning, feature engineering, a temporal
train/test split, and a versioned baseline model. It exists to show the
pipeline *working*, not to replace the unit tests in `tests/` (run those with
`pytest tests/ -v` — see `docs/test_evidence.txt` for a saved passing run).

In [1]:
import sys
sys.path.insert(0, '..')

from src import config, data_ingestion, validation, preprocessing, feature_engineering, pipeline, model
import pandas as pd
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

## 1. Ingestion

Loads the raw CSV and validates the schema before anything else touches it.

In [2]:
raw_df = data_ingestion.load_appointment_data()
print('Shape:', raw_df.shape)
raw_df.head(3)

2026-09-03 11:28:48,851 [INFO] src.data_ingestion: Loaded 5000 rows / 18 columns from /home/claude/week5/healthconnect_ml_pipeline/data/raw/HealthConnect_Appointment_Data.csv


Shape: (5000, 18)


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show


## 2. Automated Data Quality Checks

These are the Week 4 findings, turned into repeatable code (see `src/validation.py`).
Note that `waiting_time_leakage_risk` is expected to report `passed: False` — it's a
known, documented risk, not a bug; the pipeline handles it by dropping the column
in the next step rather than treating it as a hard failure.

In [3]:
quality_reports = validation.run_all_checks(raw_df)
pd.DataFrame(quality_reports)[['check', 'passed']]

2026-09-03 11:28:48,883 [INFO] src.validation: Data quality check 'no_show_history_consistency': {'check': 'no_show_history_consistency', 'passed': True, 'n_violations': 0, 'violation_ids': []}


2026-09-03 11:28:48,884 [INFO] src.validation: Data quality check 'booking_lead_days_consistency': {'check': 'booking_lead_days_consistency', 'passed': True, 'n_violations': 0, 'violation_ids': []}


2026-09-03 11:28:48,884 [INFO] src.validation: Data quality check 'appointment_day_consistency': {'check': 'appointment_day_consistency', 'passed': True, 'n_violations': 0, 'violation_ids': []}


2026-09-03 11:28:48,885 [INFO] src.validation: Data quality check 'reminder_channel_missingness_is_logical': {'check': 'reminder_channel_missingness_is_logical', 'passed': True, 'n_violations': 0, 'violation_ids': []}


2026-09-03 11:28:48,886 [WARNING] src.validation: Data quality check 'waiting_time_leakage_risk': {'check': 'waiting_time_leakage_risk', 'passed': False, 'pct_no_show_rows_with_waiting_time': 0.9847, 'note': 'waiting_time_minutes is populated for the majority of No-Show rows, which is logically inconsistent and confirms this column must be excluded from model features (see config.LEAKAGE_COLUMNS).'}


,check,passed
0,no_show_history_consistency,True
1,booking_lead_days_consistency,True
2,appointment_day_consistency,True
3,reminder_channel_missingness_is_logical,True
4,waiting_time_leakage_risk,False


In [4]:
leak_report = [r for r in quality_reports if r['check'] == 'waiting_time_leakage_risk'][0]
print(leak_report['note'])
print(f"Populated for {leak_report['pct_no_show_rows_with_waiting_time']:.1%} of No-Show rows")

waiting_time_minutes is populated for the majority of No-Show rows, which is logically inconsistent and confirms this column must be excluded from model features (see config.LEAKAGE_COLUMNS).
Populated for 98.5% of No-Show rows


## 3. Cleaning & Preprocessing

Scopes the data to the binary problem (drops Cancelled), removes the leakage
column, encodes the logical `reminder_channel` missingness, imputes the small
amount of genuine missingness in `distance_to_clinic_km`, and encodes the target.

In [5]:
clean_df = preprocessing.clean(raw_df)
print('Shape after cleaning:', clean_df.shape)
print('Outcome classes remaining:', clean_df['appointment_outcome'].unique())
print('Target balance:')
clean_df['target'].value_counts(normalize=True).round(3)

2026-09-03 11:28:48,903 [INFO] src.preprocessing: filter_to_modelling_scope: kept 4737/5000 rows (excluded classes: ['Cancelled'])


2026-09-03 11:28:48,904 [INFO] src.preprocessing: drop_leakage_columns: removed ['waiting_time_minutes']


2026-09-03 11:28:48,906 [INFO] src.preprocessing: drop_redundant_columns: removed ['appointment_day']


2026-09-03 11:28:48,909 [INFO] src.preprocessing: impute_genuine_missing_values: filled 86 missing distance_to_clinic_km values with median=8.70


Shape after cleaning: (4737, 18)
Outcome classes remaining: <StringArray>
['No-Show', 'Attended']
Length: 2, dtype: str
Target balance:


target
1    0.512
0    0.488
Name: proportion, dtype: float64

## 4. Feature Engineering

The key engineered feature is `historical_no_show_rate`: each patient's no-show
rate computed using only their *prior* appointments, to avoid leaking future
information. See `tests/test_feature_engineering.py` for a hand-verified
correctness test of this logic.

In [6]:
features_df = feature_engineering.engineer_features(clean_df)
cols_to_show = ['patient_id', 'appointment_date', 'target',
                'historical_appointment_count', 'historical_no_show_rate']
features_df.sort_values(['patient_id', 'appointment_date'])[cols_to_show].head(10)

2026-09-03 11:28:48,928 [INFO] src.feature_engineering: add_historical_no_show_rate: computed for 4737 rows across 1680 patients


,patient_id,appointment_date,target,historical_appointment_count,historical_no_show_rate
0,P-0001,2025-06-13,0,0,0.0
1,P-0001,2026-01-26,0,1,0.0
2,P-0002,2025-03-06,1,0,0.0
3,P-0002,2026-05-23,1,1,1.0
4,P-0003,2025-06-04,0,0,0.0
5,P-0003,2026-02-01,1,1,0.0
6,P-0004,2025-02-14,0,0,0.0
7,P-0004,2025-12-13,1,1,0.0
8,P-0004,2026-01-17,1,2,0.5
9,P-0007,2025-02-15,1,0,0.0


In [7]:
# Sanity check: does the engineered feature actually carry signal?
import numpy as np
bins = pd.cut(features_df['historical_appointment_count'], bins=[-1, 0, 1, 2, 100],
              labels=['0 prior', '1 prior', '2 prior', '3+ prior'])
features_df.groupby(bins, observed=True)['target'].mean().round(3)

historical_appointment_count
0 prior     0.521
1 prior     0.512
2 prior     0.507
3+ prior    0.496
Name: target, dtype: float64

The relationship holds up after the leak-safe recomputation: more prior
no-shows still associates with a higher chance of the current appointment
also being a no-show — this feature is doing real work, not just repeating
`previous_no_shows` from the raw data.

## 5. Encoding & Temporal Train/Test Split

Categorical columns are one-hot encoded, then the data is split **chronologically**
(not randomly) at `config.SPLIT_DATE`, since in production the model always
predicts future appointments from past data.

In [8]:
encoded_df = preprocessing.encode_categoricals(features_df)
train_df, test_df = pipeline.temporal_train_test_split(encoded_df)
print(f'Split date: {config.SPLIT_DATE}')
print(f'Train: {len(train_df)} rows | Test: {len(test_df)} rows')
print(f'Train date range: {train_df["appointment_date"].min()} to {train_df["appointment_date"].max()}')
print(f'Test date range:  {test_df["appointment_date"].min()} to {test_df["appointment_date"].max()}')

Split date: 2026-02-01
Train: 3406 rows | Test: 1331 rows
Train date range: 2025-01-01 00:00:00 to 2026-01-31 00:00:00
Test date range:  2026-02-01 00:00:00 to 2026-06-30 00:00:00


## 6. Full Pipeline (single call)

Everything above is also available as a single orchestrated call, with
try/except error handling wrapping each stage (see `src/pipeline.py`).

In [9]:
result = pipeline.run_pipeline(save_outputs=True)
print('Train:', result['train'].shape, '| Test:', result['test'].shape)
print('Saved to:', config.PROCESSED_DIR)

2026-09-03 11:28:49,006 [INFO] src.data_ingestion: Loaded 5000 rows / 18 columns from /home/claude/week5/healthconnect_ml_pipeline/data/raw/HealthConnect_Appointment_Data.csv


2026-09-03 11:28:49,017 [INFO] src.validation: Data quality check 'no_show_history_consistency': {'check': 'no_show_history_consistency', 'passed': True, 'n_violations': 0, 'violation_ids': []}


2026-09-03 11:28:49,018 [INFO] src.validation: Data quality check 'booking_lead_days_consistency': {'check': 'booking_lead_days_consistency', 'passed': True, 'n_violations': 0, 'violation_ids': []}


2026-09-03 11:28:49,018 [INFO] src.validation: Data quality check 'appointment_day_consistency': {'check': 'appointment_day_consistency', 'passed': True, 'n_violations': 0, 'violation_ids': []}


2026-09-03 11:28:49,019 [INFO] src.validation: Data quality check 'reminder_channel_missingness_is_logical': {'check': 'reminder_channel_missingness_is_logical', 'passed': True, 'n_violations': 0, 'violation_ids': []}


...

2026-09-03 11:28:49,029 [INFO] src.preprocessing: impute_genuine_missing_values: filled 86 missing distance_to_clinic_km values with median=8.70


2026-09-03 11:28:49,040 [INFO] src.feature_engineering: add_historical_no_show_rate: computed for 4737 rows across 1680 patients


2026-09-03 11:28:49,085 [INFO] src.pipeline: Saved processed train/test sets to /home/claude/week5/healthconnect_ml_pipeline/data/processed


2026-09-03 11:28:49,087 [INFO] src.pipeline: Pipeline complete: train=3406 rows, test=1331 rows, 31 features


Train: (3406, 31) | Test: (1331, 31)
Saved to: /home/claude/week5/healthconnect_ml_pipeline/data/processed


## 7. Baseline Model: Train, Evaluate, Save (versioned)

A Logistic Regression baseline — the goal at this stage is a working,
versioned integration, not the best-performing model.

In [10]:
trained_model, feature_cols = model.train_baseline_model(result['train'])
metrics = model.evaluate_model(trained_model, result['test'], feature_cols)
metrics

2026-09-03 11:28:49,277 [INFO] src.model: Trained baseline LogisticRegression on 3406 rows, 25 features


2026-09-03 11:28:49,290 [INFO] src.model: Baseline evaluation: {'accuracy': 0.6153, 'precision': 0.6331, 'recall': 0.598, 'f1': 0.615, 'roc_auc': 0.6635, 'confusion_matrix': [[410, 237], [275, 409]], 'n_test_rows': 1331}


{'accuracy': 0.6153,
 'precision': 0.6331,
 'recall': 0.598,
 'f1': 0.615,
 'roc_auc': 0.6635,
 'confusion_matrix': [[410, 237], [275, 409]],
 'n_test_rows': 1331}

In [11]:
model_path = model.save_model(trained_model, feature_cols, metrics)
print('Model artifact:', model_path)
print('Metadata sidecar:', model_path.with_suffix('.json'))

2026-09-03 11:28:49,296 [INFO] src.model: Saved model to /home/claude/week5/healthconnect_ml_pipeline/models/baseline_logreg_v20260903_112849.joblib (metadata: /home/claude/week5/healthconnect_ml_pipeline/models/baseline_logreg_v20260903_112849.json)


Model artifact: /home/claude/week5/healthconnect_ml_pipeline/models/baseline_logreg_v20260903_112849.joblib
Metadata sidecar: /home/claude/week5/healthconnect_ml_pipeline/models/baseline_logreg_v20260903_112849.json


## 8. Load the Saved Model Back (round-trip check)

Confirms the save/load integration actually works, not just that `.fit()` runs.

In [12]:
reloaded = model.load_model(model_path)
sample = result['test'][feature_cols].head(5)
print('Predictions from reloaded model:', reloaded.predict(sample).tolist())
print('Predictions from original model: ', trained_model.predict(sample).tolist())
assert (reloaded.predict(sample) == trained_model.predict(sample)).all()
print('Round-trip OK: reloaded model produces identical predictions.')

Predictions from reloaded model: [1, 1, 0, 0, 1]
Predictions from original model:  [1, 1, 0, 0, 1]
Round-trip OK: reloaded model produces identical predictions.


## 9. Implementation Issues Encountered

- **numpy bool vs Python bool**: the initial version of
  `check_waiting_time_leakage_risk` returned a numpy `bool_` for `"passed"`,
  which failed an `is False` identity check in a unit test even though the
  value was logically correct. Fixed by explicitly casting to `bool(...)`.
  Caught by the test suite before it could reach the pipeline output.
- **Choosing the split date**: the first candidate split date produced a
  training set that was too small relative to test. Resolved by checking the
  resulting train/test proportions for a few candidate dates before fixing
  `config.SPLIT_DATE`.
- **Historical no-show rate correctness**: the naive approach (a simple
  per-patient average) would have leaked future outcomes into earlier rows.
  Resolved with an explicit sort-by-date-then-cumulative-shift approach,
  verified against a hand-computed expected result in
  `tests/test_feature_engineering.py`.

## 10. Next Steps (Week 6)

- Resolve the `waiting_time_minutes` anomaly with the data team — if it turns
  out to be simulation noise rather than a real leakage risk, this should be
  documented and the column formally retired (not silently reconsidered).
- Compare the baseline against at least one ensemble model, still under the
  same temporal split, before selecting a candidate for further tuning.
- Add lightweight monitoring hooks (log inputs/outputs of each batch run) as
  outlined in the Week 4 architecture, ahead of any real integration with a
  reminder workflow.